# Reconstructor: VGGT-Omega

---
---
---

# Step 0: Environment Context
Force the working directory to the repository root to ensure configuration files and submodules resolve correctly.

In [ ]:
import os
import sys
from pathlib import Path

# Force the working directory to the repository root
if Path.cwd().name == "notebooks":
    os.chdir("..")

# Expose VGGT-Omega modules
vggt_path = Path("external/vggt-omega").resolve()
if str(vggt_path) not in sys.path:
    sys.path.append(str(vggt_path))

# Verify context is now repository root
print(f"Current Working Directory: {Path.cwd()}")

---
---
---

# Step 1: Initialize model and load weights

In [ ]:
import torch
from vggt_omega.models import VGGTOmega
from vggt_omega.utils.load_fn import load_and_preprocess_images
from vggt_omega.utils.pose_enc import encoding_to_camera

checkpoint_path = "external/vggt-omega/checkpoints/vggt_omega_1b_512.pt"

# Initialize and load weights
model = VGGTOmega().to("cuda").eval()
model.load_state_dict(torch.load(checkpoint_path, map_location="cpu", weights_only=True))

---
---
---

# Step 2: Choose Input Sequence

## DATASET: ScanNet

---

### scene0000_00:
sceneType = Apartment

numColorFrames = 5578

In [ ]:
input_sequence_dir = Path("/storage/group/dataset_mirrors/scannet/scans/scene0000_00/color")
output_results_dir_base = Path("data/reconstruction/vggt_omega_output/scannet/scene0000_00")

---

### scene0001_00:
sceneType = Living room / Lounge

numColorFrames = 1545

In [ ]:
input_sequence_dir = Path("/storage/group/dataset_mirrors/scannet/scans/scene0001_00/color")
output_results_dir_base = Path("data/reconstruction/vggt_omega_output/scannet/scene0001_00")

---

### scene0002_00:
sceneType = Living room / Lounge

numColorFrames = 5193

In [ ]:
input_sequence_dir = Path("/storage/group/dataset_mirrors/scannet/scans/scene0002_00/color")
output_results_dir_base = Path("data/reconstruction/vggt_omega_output/scannet/scene0002_00")

---

### scene0003_00:
sceneType = Kitchen

numColorFrames = 1736

In [ ]:
input_sequence_dir = Path("/storage/group/dataset_mirrors/scannet/scans/scene0003_00/color")
output_results_dir_base = Path("data/reconstruction/vggt_omega_output/scannet/scene0003_00")

---

### scene0004_00:
sceneType = Hallway

numColorFrames = 929

In [ ]:
input_sequence_dir = Path("/storage/group/dataset_mirrors/scannet/scans/scene0004_00/color")
output_results_dir_base = Path("data/reconstruction/vggt_omega_output/scannet/scene0004_00")

---
---
---

# Step 3: Generate custom config for target sequence

In [ ]:
config_number = "01"
config_filename = f"custom_3DRoomSearch_v{config_number}.yaml"

In [ ]:
abs_output_dir = (output_results_dir_base / f"config_{config_number}").resolve()
abs_output_dir.mkdir(parents=True, exist_ok=True)
abs_config_path = (abs_output_dir / f"{config_filename}").resolve()

In [ ]:
stride = 10
max_frames = 100

In [ ]:
# Parse sequence and apply strict numerical sorting
all_images = sorted(
    list(input_sequence_dir.glob("*.jpg")), 
    key=lambda x: int(x.stem)
)

sampled_paths = all_images[::stride][:max_frames]
image_names = [str(p) for p in sampled_paths]
num_images = len(image_names)

print(f"Sampled {num_images} frames (Stride: {stride}) for inference.")

### (optional): Visualize resulting input sequence

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

print(f"Visualizing {num_images} frames...")

# Render geometry parameters
max_cols = 10
cols = min(num_images, max_cols)
rows = (num_images + max_cols - 1) // max_cols

# squeeze=False ensures axes is always a 2D array, enabling safe flattening
fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows), squeeze=False)
axes = axes.flatten()

for i, ax in enumerate(axes):
    if i < num_images:
        img = Image.open(image_names[i])
        ax.imshow(img)
        ax.set_title(Path(image_names[i]).name, fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

### Save config

In [ ]:
# Define the configuration as a multi-line Python f-string
config_content = f"""# ViSTA-SLAM Custom Configuration v{config_number}
stride: {stride}
max_frames: {max_frames}
"""

# Write the string directly to the absolute path
abs_config_path.write_text(config_content)

print(f"✅ Generated custom configuration: {abs_config_path.name}")

---
---
---

# Step 4: Invoke VGGT-Omega

In [ ]:
# Load and preprocess using the correct, numerically sorted frame paths
images = load_and_preprocess_images(image_names, image_resolution=512).to("cuda")

print(f"Preprocessed tensor shape: {images.shape}")  # Expected: [B, C, H, W]

In [ ]:
with torch.inference_mode():
    predictions = model(images)

# Decode camera parameters and append directly to the predictions dictionary
extrinsics, intrinsics = encoding_to_camera(
    predictions["pose_enc"],
    predictions["images"].shape[-2:],
)
predictions["extrinsics"] = extrinsics
predictions["intrinsics"] = intrinsics

print(f"Successfully processed {len(image_names)} frames.")
print(f"Available prediction outputs: {list(predictions.keys())}")

### (optional): Check shapes

In [ ]:
print("--- Raw Tensor Shapes ---")
print(f"Depth:      {predictions['depth'].shape}")
print(f"Confidence: {predictions['depth_conf'].shape}")
print(f"Extrinsics: {predictions['extrinsics'].shape}")
print(f"Intrinsics: {predictions['intrinsics'].shape}")
print(f"Sequence:   {len(image_names)} frames")

---
---
---

# Step 5: Store Tensors and Metadata

In [ ]:
import json
import torch

output_tensors_file = abs_output_dir / "vggt_predictions.pt"
output_names_file = abs_output_dir / "image_names.json"

# Move predictions to CPU once
depth_cpu = predictions["depth"].cpu()
depth_conf_cpu = predictions["depth_conf"].cpu()
ext_cpu = predictions["extrinsics"].cpu()
int_cpu = predictions["intrinsics"].cpu()

# Store model tensor outputs
tensor_dict = {
    "depth": depth_cpu,
    "depth_conf": depth_conf_cpu,
    "extrinsics": ext_cpu,
    "intrinsics": int_cpu,
}
torch.save(tensor_dict, output_tensors_file)
print(f"Saved tensor predictions to: {output_tensors_file.name}")

# Store sequence metadata separately
with open(output_names_file, "w") as f:
    json.dump(image_names, f, indent=4)
print(f"Saved image names to: {output_names_file.name}")

---
---
---

# Step 6: Point Cloud Fusion and Export

In [ ]:
conf_thresh = 0.5

In [ ]:
import numpy as np
import open3d as o3d
from PIL import Image
import torch

output_pcd_file = abs_output_dir / "pointcloud.ply"

# 1. Source tensors directly from GPU predictions
# Strip Batch dimension (0) and trailing Channel dimension (-1) from depth
depths = predictions["depth"].squeeze(-1)[0]       # [N, H, W] on CUDA
confs = predictions["depth_conf"][0]               # [N, H, W] on CUDA
exts = predictions["extrinsics"][0]                # [N, 3, 4] on CUDA
ints = predictions["intrinsics"][0]                # [N, 3, 3] on CUDA

N, H, W = depths.shape

# 2. Allocate unprojection grid natively on GPU
v, u = torch.meshgrid(
    torch.arange(H, device="cuda"), 
    torch.arange(W, device="cuda"), 
    indexing="ij"
)
uv = torch.stack([u, v], dim=-1).float()           # [H, W, 2] on CUDA

global_points = []
global_colors = []

for i in range(N):
    depth = depths[i]
    conf = confs[i]
    K = ints[i]
    T_cw = exts[i]

    # Pad to 4x4 and invert to Camera-to-World on GPU
    T_cw_4x4 = torch.eye(4, device="cuda")
    T_cw_4x4[:3, :4] = T_cw
    T_wc = torch.linalg.inv(T_cw_4x4)[:3, :4]

    # Filter invalid/low-confidence geometry
    mask = (conf > conf_thresh) & (depth > 0)
    z = depth[mask]
    uv_valid = uv[mask]

    # Pinhole unprojection to local frame C: Pc = [X, Y, Z, 1]^T
    x = (uv_valid[:, 0] - K[0, 2]) * z / K[0, 0]
    y = (uv_valid[:, 1] - K[1, 2]) * z / K[1, 1]
    pts_local = torch.stack([x, y, z, torch.ones_like(z)], dim=-1) # [M, 4]

    # SE(3) transformation to global frame W
    pts_global = (T_wc @ pts_local.T).T
    
    # 3. Transfer ONLY the highly filtered 3D coordinates to CPU
    global_points.append(pts_global.cpu())

    # Extract aligned RGB values (Read via PIL directly to CPU)
    img = np.array(Image.open(image_names[i]).resize((W, H)))
    colors = torch.from_numpy(img).float() / 255.0
    
    # Apply GPU mask to CPU colors
    global_colors.append(colors[mask.cpu()])

# Fuse and export
all_points = torch.cat(global_points, dim=0).numpy()
all_colors = torch.cat(global_colors, dim=0).numpy()

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(all_points)
pcd.colors = o3d.utility.Vector3dVector(all_colors)

o3d.io.write_point_cloud(str(output_pcd_file), pcd)
print(f"✅ Fused pointcloud saved to: {output_pcd_file.name}")

---
---
---